# 10. Corrected policy optimization

This notebook produces the terminally corrected comparison between the dynamic and myopic promotion policies.

The implementation uses:

- the same calibrated behavioral draws for optimization and evaluation;
- product-specific, empirically supported promotion actions;
- a 12-week decision horizon followed by a no-promotion washout;
- weekly demand, regular-price, and cost profiles from the held-out period;
- category-level weekly promotion capacities;
- exact product- and week-level accounting decompositions.

The no-washout results are retained only to quantify the terminal-boundary artifact.

## 1. Imports, paths, and configuration

In [ ]:
from __future__ import annotations

from dataclasses import replace
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        root
        for root in [CURRENT_DIR, CURRENT_DIR.parent]
        if (root / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root. Run this notebook from the "
        "repository root or the notebooks directory."
    )

for import_directory in [
    PROJECT_ROOT / "src",
    PROJECT_ROOT,
    CURRENT_DIR,
]:
    if import_directory.is_dir() and str(import_directory) not in sys.path:
        sys.path.insert(0, str(import_directory))

from corrected_promotion_analysis import (
    PlanningSpec,
    SupportSpec,
    build_schedule_system,
    build_weekly_economic_profiles,
    coerce_action_sets,
    constant_weekly_profiles,
    load_pickle,
    prepare_behavioral_draws,
    prepare_support_table,
    run_policy_grid,
    save_pickle,
    schedule_input_fingerprint,
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLE_DIR = PROJECT_ROOT / "results" / "tables"
FIGURE_DIR = PROJECT_ROOT / "results" / "figures"

for directory in [PROCESSED_DIR, TABLE_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

In [ ]:
# Core empirical design.
DECISION_HORIZON = 12
WASHOUT_HORIZONS = (0, 4, 6, 8, 10, 12)
MAXIMUM_WASHOUT = max(WASHOUT_HORIZONS)

COOLDOWN = 2
MAX_PROMOTIONS = 4
DISCOUNT_FACTOR = 0.995

# Funding and category capacity.
MAIN_ALPHA = 2.24
ALPHA_GRID = np.unique(
    np.round(
        np.concatenate(
            [
                np.arange(1.40, 3.21, 0.05),
                np.array([MAIN_ALPHA]),
            ]
        ),
        4,
    )
)
CAPACITIES = (1, 2, 3, 8)

# Empirical support baseline.
BASELINE_SUPPORT = SupportSpec(
    bin_width=0.05,
    minimum_depth=0.03,
    maximum_depth=0.60,
    minimum_observations=15,
    minimum_panels=3,
    maximum_positive_actions=3,
    matching_tolerance=0.03,
)

# Weekly economic profiles.
USE_WEEKLY_ECONOMICS = True
START_TEST_WEEK_OFFSET = 0

# Numerical controls.
SCHEDULE_BATCH_SIZE = 256
COMPUTE_SECOND_BEST_ON_FULL_GRID = True
MILP_TIME_LIMIT_SECONDS = None

# Washout-selection diagnostics.
VDO_STABILITY_TOLERANCE = 0.05
TERMINAL_STATE_TOLERANCE = 0.01

print("Alpha values:", len(ALPHA_GRID))
print("Capacities:", CAPACITIES)
print("Washout horizons:", WASHOUT_HORIZONS)

## 2. Load behavioral draws, support information, and held-out economics

In [ ]:
PRODUCT_DRAW_PATH = (
    PROCESSED_DIR
    / "product_stockpiling_behavioral_draws_v2.pkl"
)
PRODUCT_ACTION_SUPPORT_PATH = (
    TABLE_DIR
    / "product_supported_action_clusters.csv"
)
PRODUCT_ACTION_SET_PATH = (
    PROCESSED_DIR
    / "product_supported_action_sets.pkl"
)

SELECTED_SAMPLE_PATH = (
    PROCESSED_DIR
    / "paper_selected_sample.parquet"
)
DEMAND_PREDICTION_PATH = (
    PROCESSED_DIR
    / "final_demand_model_predictions.pkl"
)

required_paths = [
    PRODUCT_DRAW_PATH,
    PRODUCT_ACTION_SUPPORT_PATH,
    PRODUCT_ACTION_SET_PATH,
]

missing = [path for path in required_paths if not path.is_file()]
if missing:
    raise FileNotFoundError(
        "Required upstream artifacts are missing:\n"
        + "\n".join(f"  - {path}" for path in missing)
    )

raw_draws = pd.read_pickle(PRODUCT_DRAW_PATH)
draw_frame, draws_by_product = prepare_behavioral_draws(raw_draws)

raw_support_table = pd.read_csv(PRODUCT_ACTION_SUPPORT_PATH)
support_table = prepare_support_table(raw_support_table)

raw_action_artifact = pd.read_pickle(PRODUCT_ACTION_SET_PATH)
products = sorted(draws_by_product)
action_sets = coerce_action_sets(raw_action_artifact, products)

print("Products:", len(products))
print("Behavioral draws:", len(draw_frame))
print("Supported positive actions:", sum(len(values) - 1 for values in action_sets.values()))

display(
    pd.DataFrame(
        {
            "upc": products,
            "actions": [
                ", ".join(f"{100 * value:.0f}%" for value in action_sets[upc])
                for upc in products
            ],
            "draws": [
                len(draws_by_product[upc]["weights"])
                for upc in products
            ],
        }
    )
)

In [ ]:
if USE_WEEKLY_ECONOMICS:
    weekly_required = [
        SELECTED_SAMPLE_PATH,
        DEMAND_PREDICTION_PATH,
    ]
    weekly_missing = [
        path for path in weekly_required
        if not path.is_file()
    ]

    if weekly_missing:
        raise FileNotFoundError(
            "Weekly economics were requested, but the following files "
            "are missing:\n"
            + "\n".join(f"  - {path}" for path in weekly_missing)
        )

    selected_sample = pd.read_parquet(
        SELECTED_SAMPLE_PATH
    )
    demand_predictions = pd.read_pickle(
        DEMAND_PREDICTION_PATH
    )

    (
        weekly_profile_table,
        weekly_profiles,
        source_weeks,
    ) = build_weekly_economic_profiles(
        selected_sample=selected_sample,
        demand_predictions=demand_predictions,
        products=products,
        decision_horizon=DECISION_HORIZON,
        maximum_washout=MAXIMUM_WASHOUT,
        start_test_week_offset=START_TEST_WEEK_OFFSET,
        model_name="product_promotion",
    )
    ECONOMIC_PROFILE_MODE = "held_out_weekly_profiles"

else:
    (
        weekly_profile_table,
        weekly_profiles,
        source_weeks,
    ) = constant_weekly_profiles(
        products=products,
        total_weeks=DECISION_HORIZON + MAXIMUM_WASHOUT,
    )
    ECONOMIC_PROFILE_MODE = "constant_product_economics"

print("Economic profile mode:", ECONOMIC_PROFILE_MODE)
print("Source weeks:", source_weeks)
display(weekly_profile_table.head(24))

The weekly profiles are multiplicative adjustments to the calibrated product-level economic levels in the behavioral-draw artifact. This preserves the original calibration scale while allowing baseline demand, regular prices, and costs to vary over the evaluation weeks. The demand profile is estimated from held-out PPML predictions in regular, non-post-promotion observations.

## 3. Build and cache schedule systems for each washout horizon

In [ ]:
SCHEDULE_CACHE_TEMPLATE = (
    PROCESSED_DIR
    / "corrected_schedule_system_w{washout}.pkl"
)

schedule_systems = {}

for washout_horizon in WASHOUT_HORIZONS:
    planning = PlanningSpec(
        decision_horizon=DECISION_HORIZON,
        washout_horizon=washout_horizon,
        cooldown=COOLDOWN,
        max_promotions=MAX_PROMOTIONS,
        discount_factor=DISCOUNT_FACTOR,
        alpha_min=float(ALPHA_GRID.min()),
        alpha_max=float(ALPHA_GRID.max()),
    )

    expected_fingerprint = schedule_input_fingerprint(
        draws_by_product=draws_by_product,
        weekly_profiles=weekly_profiles,
        action_sets=action_sets,
        planning=planning,
        alpha_grid=ALPHA_GRID,
    )

    cache_path = Path(
        str(SCHEDULE_CACHE_TEMPLATE).format(
            washout=washout_horizon
        )
    )

    cache_is_current = False

    if cache_path.is_file():
        cached_system = load_pickle(
            cache_path
        )
        cache_is_current = (
            cached_system.get(
                "input_fingerprint"
            )
            == expected_fingerprint
        )

    if cache_is_current:
        schedule_system = cached_system
        print(
            f"Loaded current schedule cache: "
            f"washout={washout_horizon}"
        )

    else:
        schedule_system = build_schedule_system(
            draws_by_product=draws_by_product,
            weekly_profiles=weekly_profiles,
            action_sets=action_sets,
            planning=planning,
            alpha_grid=ALPHA_GRID,
            batch_size=SCHEDULE_BATCH_SIZE,
        )

        save_pickle(
            schedule_system,
            cache_path,
        )

        print(
            f"Built schedule system: "
            f"washout={washout_horizon}"
        )

    schedule_systems[washout_horizon] = (
        schedule_system
    )

candidate_rows = []

for washout_horizon, schedule_system in (
    schedule_systems.items()
):
    for upc in schedule_system["products"]:
        artifact = (
            schedule_system[
                "product_artifacts"
            ][upc]
        )
        candidate_rows.append(
            {
                "washout_horizon": (
                    washout_horizon
                ),
                "upc": upc,
                "feasible_schedules": len(
                    artifact["schedules"]
                ),
                "retained_candidates": len(
                    artifact["candidates"]
                ),
            }
        )

candidate_summary = pd.DataFrame(
    candidate_rows
)

display(
    candidate_summary.groupby(
        "washout_horizon",
        observed=True,
    )[
        [
            "feasible_schedules",
            "retained_candidates",
        ]
    ].sum()
)

## 4. Washout stabilization at the main funding value

In [ ]:
washout_runs = []

for washout_horizon in WASHOUT_HORIZONS:
    print(
        f"Running washout={washout_horizon} "
        f"at alpha={MAIN_ALPHA:.2f}"
    )

    run = run_policy_grid(
        schedule_system=(
            schedule_systems[
                washout_horizon
            ]
        ),
        draws_by_product=draws_by_product,
        weekly_profiles=weekly_profiles,
        action_sets=action_sets,
        support_table=support_table,
        alpha_values=[MAIN_ALPHA],
        capacities=CAPACITIES,
        compute_second_best=False,
        time_limit_seconds=(
            MILP_TIME_LIMIT_SECONDS
        ),
    )

    run["results"][
        "economic_profile_mode"
    ] = ECONOMIC_PROFILE_MODE

    washout_runs.append(
        run
    )

washout_results = pd.concat(
    [
        run["results"]
        for run in washout_runs
    ],
    ignore_index=True,
).sort_values(
    [
        "capacity",
        "washout_horizon",
    ]
)

washout_results[
    "vdo_change_from_previous"
] = (
    washout_results.groupby(
        "capacity",
        observed=True,
    )[
        "vdo"
    ].diff()
)

display(
    washout_results[
        [
            "capacity",
            "washout_horizon",
            "dynamic_profit",
            "myopic_profit",
            "vdo",
            "vdo_percent",
            "vdo_change_from_previous",
            "maximum_terminal_state_dynamic",
            "maximum_terminal_state_myopic",
        ]
    ]
)

In [ ]:
stability_rows = []

for washout_horizon in sorted(
    value
    for value in WASHOUT_HORIZONS
    if value > 0
):
    current = washout_results.loc[
        washout_results[
            "washout_horizon"
        ].eq(
            washout_horizon
        )
    ].copy()

    maximum_vdo_change = float(
        current[
            "vdo_change_from_previous"
        ].abs().max()
    )
    maximum_terminal_state = float(
        current[
            [
                "maximum_terminal_state_dynamic",
                "maximum_terminal_state_myopic",
            ]
        ].max().max()
    )

    stability_rows.append(
        {
            "washout_horizon": (
                washout_horizon
            ),
            "maximum_absolute_vdo_change": (
                maximum_vdo_change
            ),
            "maximum_terminal_state": (
                maximum_terminal_state
            ),
            "vdo_stable": (
                maximum_vdo_change
                <= VDO_STABILITY_TOLERANCE
            ),
            "terminal_state_small": (
                maximum_terminal_state
                <= TERMINAL_STATE_TOLERANCE
            ),
        }
    )

washout_stability = pd.DataFrame(
    stability_rows
)

eligible_stable = washout_stability.loc[
    washout_stability["vdo_stable"]
    & washout_stability[
        "terminal_state_small"
    ]
]

if eligible_stable.empty:
    SELECTED_WASHOUT = MAXIMUM_WASHOUT
    print(
        "No shorter washout satisfies both "
        "stability criteria. Using the maximum:",
        SELECTED_WASHOUT,
    )
else:
    SELECTED_WASHOUT = int(
        eligible_stable[
            "washout_horizon"
        ].min()
    )
    print(
        "Selected stabilized washout:",
        SELECTED_WASHOUT,
    )

display(washout_stability)

In [ ]:
fig, ax = plt.subplots(
    figsize=(8.8, 4.8)
)

for capacity, group in (
    washout_results.groupby(
        "capacity",
        observed=True,
    )
):
    ax.plot(
        group["washout_horizon"],
        group["vdo"],
        marker="o",
        label=f"B={capacity}",
    )

ax.axhline(
    0.0,
    linestyle=":",
    linewidth=1.0,
)
ax.axvline(
    SELECTED_WASHOUT,
    linestyle="--",
    linewidth=1.0,
)
ax.set_xlabel(
    "Washout horizon (weeks)"
)
ax.set_ylabel(
    "Value of dynamic optimization"
)
ax.set_title(
    "Terminal-washout stabilization"
)
ax.legend(
    title="Weekly capacity"
)
fig.tight_layout()

washout_figure_png = (
    FIGURE_DIR
    / "corrected_vdo_washout_stability.png"
)
washout_figure_pdf = (
    FIGURE_DIR
    / "corrected_vdo_washout_stability.pdf"
)

fig.savefig(
    washout_figure_png,
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    washout_figure_pdf,
    bbox_inches="tight",
)

plt.show()

## 5. Full funding-capacity policy grid at the stabilized washout

In [ ]:
primary_schedule_system = (
    schedule_systems[
        SELECTED_WASHOUT
    ]
)

primary_run = run_policy_grid(
    schedule_system=(
        primary_schedule_system
    ),
    draws_by_product=draws_by_product,
    weekly_profiles=weekly_profiles,
    action_sets=action_sets,
    support_table=support_table,
    alpha_values=ALPHA_GRID,
    capacities=CAPACITIES,
    compute_second_best=(
        COMPUTE_SECOND_BEST_ON_FULL_GRID
    ),
    time_limit_seconds=(
        MILP_TIME_LIMIT_SECONDS
    ),
)

policy_results = (
    primary_run["results"]
    .copy()
    .sort_values(
        [
            "capacity",
            "alpha",
        ]
    )
    .reset_index(drop=True)
)

policy_results[
    "economic_profile_mode"
] = ECONOMIC_PROFILE_MODE

display(
    policy_results.loc[
        np.isclose(
            policy_results["alpha"],
            MAIN_ALPHA,
        )
    ][
        [
            "alpha",
            "capacity",
            "dynamic_profit",
            "myopic_profit",
            "vdo",
            "vdo_percent",
            "dynamic_promotion_count",
            "myopic_promotion_count",
            "dynamic_binding_weeks",
            "myopic_binding_weeks",
            "action_disagreements",
            "best_second_gap",
        ]
    ]
)

In [ ]:
fig, ax = plt.subplots(
    figsize=(9.2, 5.0)
)

for capacity, group in (
    policy_results.groupby(
        "capacity",
        observed=True,
    )
):
    ax.plot(
        group["alpha"],
        group["vdo"],
        label=f"B={capacity}",
    )

ax.axhline(
    0.0,
    linestyle=":",
    linewidth=1.0,
)
ax.axvline(
    MAIN_ALPHA,
    linestyle="--",
    linewidth=1.0,
)
ax.set_xlabel(
    r"Contract generosity, $\alpha$"
)
ax.set_ylabel(
    "Value of dynamic optimization"
)
ax.set_title(
    "Terminally corrected value of dynamic planning"
)
ax.legend(
    title="Weekly capacity"
)
fig.tight_layout()

policy_figure_png = (
    FIGURE_DIR
    / "corrected_vdo_by_alpha_capacity.png"
)
policy_figure_pdf = (
    FIGURE_DIR
    / "corrected_vdo_by_alpha_capacity.pdf"
)

fig.savefig(
    policy_figure_png,
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    policy_figure_pdf,
    bbox_inches="tight",
)

plt.show()

## 6. Save standardized artifacts for Notebook 11

In [ ]:
CORRECTED_ARTIFACT_PATH = (
    PROCESSED_DIR
    / "corrected_policy_optimization_artifact.pkl"
)

POLICY_RESULTS_PATH = (
    TABLE_DIR
    / "corrected_policy_results.csv"
)
WASHOUT_RESULTS_PATH = (
    TABLE_DIR
    / "corrected_policy_washout_results.csv"
)
WASHOUT_STABILITY_PATH = (
    TABLE_DIR
    / "corrected_policy_washout_stability.csv"
)
PRODUCT_DECOMPOSITION_PATH = (
    TABLE_DIR
    / "corrected_policy_product_decomposition.csv"
)
WEEKLY_DECOMPOSITION_PATH = (
    TABLE_DIR
    / "corrected_policy_weekly_decomposition.csv"
)
WEEKLY_PROFILE_PATH = (
    TABLE_DIR
    / "corrected_weekly_economic_profiles.csv"
)
CANDIDATE_SUMMARY_PATH = (
    TABLE_DIR
    / "corrected_schedule_candidate_summary.csv"
)

artifact = {
    "selected_washout": (
        SELECTED_WASHOUT
    ),
    "main_alpha": (
        MAIN_ALPHA
    ),
    "alpha_grid": (
        ALPHA_GRID
    ),
    "capacities": (
        CAPACITIES
    ),
    "baseline_support": (
        BASELINE_SUPPORT
    ),
    "economic_profile_mode": (
        ECONOMIC_PROFILE_MODE
    ),
    "source_weeks": (
        source_weeks
    ),
    "products": (
        products
    ),
    "action_sets": (
        action_sets
    ),
    "support_table": (
        support_table
    ),
    "weekly_profile_table": (
        weekly_profile_table
    ),
    "weekly_profiles": (
        weekly_profiles
    ),
    "draw_frame": (
        draw_frame
    ),
    "draws_by_product": (
        draws_by_product
    ),
    "schedule_system": (
        primary_schedule_system
    ),
    "policy_results": (
        policy_results
    ),
    "washout_results": (
        washout_results
    ),
    "washout_stability": (
        washout_stability
    ),
    "schedules": (
        primary_run["schedules"]
    ),
    "product_decomposition": (
        primary_run[
            "product_decomposition"
        ]
    ),
    "weekly_decomposition": (
        primary_run[
            "weekly_decomposition"
        ]
    ),
}

save_pickle(
    artifact,
    CORRECTED_ARTIFACT_PATH,
)

policy_results.to_csv(
    POLICY_RESULTS_PATH,
    index=False,
)
washout_results.to_csv(
    WASHOUT_RESULTS_PATH,
    index=False,
)
washout_stability.to_csv(
    WASHOUT_STABILITY_PATH,
    index=False,
)
primary_run[
    "product_decomposition"
].to_csv(
    PRODUCT_DECOMPOSITION_PATH,
    index=False,
)
primary_run[
    "weekly_decomposition"
].to_csv(
    WEEKLY_DECOMPOSITION_PATH,
    index=False,
)
weekly_profile_table.to_csv(
    WEEKLY_PROFILE_PATH,
    index=False,
)
candidate_summary.to_csv(
    CANDIDATE_SUMMARY_PATH,
    index=False,
)

print("Saved artifact:", CORRECTED_ARTIFACT_PATH)
print("Selected washout:", SELECTED_WASHOUT)
print("Policy rows:", len(policy_results))

## Interpretation guardrails

- Promotions may be selected only during weeks 1--12.
- The washout contains no new promotions and is included in both policies' value functions.
- Dynamic and myopic policies are evaluated using the same behavioral draws and weekly economic profiles.
- Product and week decompositions are exact accounting decompositions of predicted profit, not causal decompositions of the capacity mechanism.
- A positive VDO should be interpreted relative to myopic profit and the best--second-best schedule gap, not only in absolute units.